###Import Required Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

###Read Dataset From Volume

In [0]:
matches_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/default/ipl_data/matches.csv")

display(matches_df)

In [0]:
deliveries_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/workspace/default/ipl_data/deliveries.csv")

display(deliveries_df)

###Create BRONZE Layer Tables

In [0]:
matches_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_matches")

In [0]:
deliveries_df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("bronze_deliveries")

In [0]:
spark.sql("SELECT * FROM bronze_matches").show(5)

In [0]:
spark.sql("SELECT * FROM bronze_deliveries").show(5)

###Create Silver Matches Table (Clean Data)

In [0]:
silver_matches = spark.sql("""
SELECT
id AS match_id,
season,
team1,
team2,
toss_winner,
toss_decision,
venue,
winner
FROM bronze_matches
WHERE winner IS NOT NULL
""")

display(silver_matches)

###Save Silver Matches Table

In [0]:
silver_matches.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_matches")

###Clean Deliveries Data

In [0]:
silver_deliveries = spark.sql("""
SELECT
match_id,
inning,
batting_team,
bowling_team,
over,
ball,
batsman,
bowler,
total_runs,
player_dismissed
FROM bronze_deliveries
""")

display(silver_deliveries)

###Save Silver Deliveries Table

In [0]:
silver_deliveries.write.format("delta") \
.mode("overwrite") \
.saveAsTable("silver_deliveries")

###Verify Silver Tables

In [0]:
spark.sql("SELECT * FROM silver_matches LIMIT 5").show()

In [0]:
spark.sql("SELECT * FROM silver_deliveries LIMIT 5").show()

###Create ML Feature Dataset

In [0]:
gold_match_features = spark.sql("""

SELECT
match_id,
team1,
team2,
toss_winner,
venue,
CASE
    WHEN winner = team1 THEN 1
    ELSE 0
END AS team1_win

FROM silver_matches

""")

display(gold_match_features)


###Save Gold Feature Table

In [0]:
gold_match_features.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold_match_features")

###Verify Gold Table

In [0]:
spark.sql("SELECT * FROM gold_match_features LIMIT 5").show()

###Convert Spark Data to Pandas

In [0]:
ml_df = spark.sql("SELECT * FROM gold_match_features")

pandas_df = ml_df.toPandas()

pandas_df.head()

###Encode Categorical Columns

In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

pandas_df['team1'] = le.fit_transform(pandas_df['team1'])
pandas_df['team2'] = le.fit_transform(pandas_df['team2'])
pandas_df['toss_winner'] = le.fit_transform(pandas_df['toss_winner'])
pandas_df['venue'] = le.fit_transform(pandas_df['venue'])

###Split Dataset for Training

In [0]:
from sklearn.model_selection import train_test_split

X = pandas_df[['team1','team2','toss_winner','venue']]
y = pandas_df['team1_win']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

###Train Machine Learning Model

In [0]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()

model.fit(X_train, y_train)

###Predict Results

In [0]:
predictions = model.predict(X_test)

predictions[:10]

###Evaluate Model Accuracy

In [0]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)

###Calculate Team Win Count

In [0]:
team_wins = spark.sql("""

SELECT
winner AS team,
COUNT(*) AS total_wins
FROM silver_matches
GROUP BY winner

""")

display(team_wins)

###Calculate Total Matches Played

In [0]:
team_matches = spark.sql("""

SELECT team, COUNT(*) AS matches_played
FROM (
    SELECT team1 AS team FROM silver_matches
    UNION ALL
    SELECT team2 AS team FROM silver_matches
)
GROUP BY team

""")

display(team_matches)

###Create Team Win Percentage

In [0]:
team_stats = team_matches.join(
    team_wins,
    team_matches.team == team_wins.team,
    "left"
).select(
    team_matches.team,
    col("matches_played"),
    col("total_wins")
)

team_stats = team_stats.fillna(0)

team_stats = team_stats.withColumn(
    "win_percentage",
    col("total_wins") / col("matches_played")
)

display(team_stats)

###Join Team Stats With Matches

In [0]:
team1_stats = team_stats.withColumnRenamed("team","team1") \
.withColumnRenamed("win_percentage","team1_win_pct")

team2_stats = team_stats.withColumnRenamed("team","team2") \
.withColumnRenamed("win_percentage","team2_win_pct")

###Build Improved ML Dataset

In [0]:
enhanced_features = gold_match_features \
.join(team1_stats, "team1", "left") \
.join(team2_stats, "team2", "left")

display(enhanced_features)

###Convert to Pandas Again

In [0]:
enhanced_pd = enhanced_features.toPandas()

enhanced_pd.head()

###Encode Categorical Columns

In [0]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

enhanced_pd['team1'] = le.fit_transform(enhanced_pd['team1'])
enhanced_pd['team2'] = le.fit_transform(enhanced_pd['team2'])
enhanced_pd['toss_winner'] = le.fit_transform(enhanced_pd['toss_winner'])
enhanced_pd['venue'] = le.fit_transform(enhanced_pd['venue'])

###Train Model Again

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = enhanced_pd[['team1','team2','toss_winner','venue','team1_win_pct','team2_win_pct']]
y = enhanced_pd['team1_win']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()

model.fit(X_train, y_train)

###Predict

In [0]:
pred = model.predict(X_test)

###Check Accuracy Again

In [0]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, pred)

print("Improved Model Accuracy:", accuracy)

###Predict Win Probability

In [0]:
probabilities = model.predict_proba(X_test)

probabilities[:5]

###Convert to Percentage

In [0]:
import pandas as pd

results = pd.DataFrame({
    "Team1_Win_Probability": probabilities[:,1],
    "Actual_Result": y_test.values
})

results.head()

###Show Percentage

In [0]:
results["Team1_Win_%"] = results["Team1_Win_Probability"] * 100

results.head()

###Import MLflow

In [0]:
import mlflow
import mlflow.sklearn

###Start MLflow Experiment

In [0]:
mlflow.start_run()

###Log Model Metrics

In [0]:
mlflow.log_metric("accuracy", accuracy)

###Log the Model

In [0]:
mlflow.sklearn.log_model(
    model,
    "ipl_match_prediction_model",
    input_example=X_train.iloc[:5]
)

mlflow.end_run()

###SQL Analytics

####Top IPL Teams by Wins

In [0]:
%sql
SELECT
winner AS team,
COUNT(*) AS total_wins
FROM silver_matches
GROUP BY winner
ORDER BY total_wins DESC

####Toss Impact on Match Result

In [0]:
%sql
SELECT
toss_winner,
COUNT(*) AS matches_won
FROM silver_matches
WHERE toss_winner = winner
GROUP BY toss_winner
ORDER BY matches_won DESC

####Venue Advantage

In [0]:
%sql
SELECT
venue,
COUNT(*) AS matches_played
FROM silver_matches
GROUP BY venue
ORDER BY matches_played DESC

####Most Successful Teams

In [0]:
%sql
SELECT
team,
win_percentage
FROM (
    SELECT
    winner AS team,
    COUNT(*) AS wins,
    COUNT(*) * 1.0 /
    (SELECT COUNT(*) FROM silver_matches) AS win_percentage
    FROM silver_matches
    GROUP BY winner
)
ORDER BY win_percentage DESC

####Average Runs per Match

In [0]:
%sql
SELECT
match_id,
SUM(total_runs) AS total_runs
FROM silver_deliveries
GROUP BY match_id
ORDER BY total_runs DESC

####Best Batting Teams

In [0]:
%sql
SELECT
batting_team,
SUM(total_runs) AS total_runs
FROM silver_deliveries
GROUP BY batting_team
ORDER BY total_runs DESC

####Most Successful Teams by Venue

In [0]:
%sql
SELECT
venue,
winner,
COUNT(*) AS wins
FROM silver_matches
GROUP BY venue, winner
ORDER BY wins DESC

####Toss Decision Impact

In [0]:
%sql
SELECT
toss_decision,
COUNT(*) AS matches
FROM silver_matches
GROUP BY toss_decision

####Average Runs by Team

In [0]:
%sql
SELECT
batting_team,
AVG(total_runs) AS avg_runs
FROM silver_deliveries
GROUP BY batting_team
ORDER BY avg_runs DESC

####Top Wicket Taking Bowlers

In [0]:
%sql
SELECT
bowler,
COUNT(player_dismissed) AS wickets
FROM silver_deliveries
WHERE player_dismissed IS NOT NULL
GROUP BY bowler
ORDER BY wickets DESC

In [0]:
spark.sql("SELECT * FROM silver_matches") \
.toPandas() \
.to_csv("/Volumes/workspace/default/ipl_data/silver_matches.csv", index=False)

In [0]:
spark.sql("SELECT * FROM silver_deliveries") \
.toPandas() \
.to_csv("/Volumes/workspace/default/ipl_data/silver_deliveries.csv", index=False)